# Service Kernel Gateway — Z3 Sudoku

Premier service du **pli 0** de l'Epic #19571 : exposer le carnet phare
[Z3-02-Sudoku-Python.ipynb](../Z3-02-Sudoku-Python.ipynb) en microservice REST via
Jupyter **Kernel Gateway** (mode `notebook-http`).

**Contrat** : `POST /sudoku/solve` — corps `{"puzzle": "<81 caracteres, 0 ou . pour vide>"}`
-> `{"status": "sat"|"unsat"|"invalid", "solution": "<81 caracteres>|null", "time_ms": <float>}`.

Ce carnet s'execute aussi **hors gateway** (mode local) : il sert alors de demonstration
sur le puzzle du carnet phare. Voir le [README](README.md) pour le demarrage.

In [1]:
# Cellule de preparation — executee une fois au demarrage du kernel.
import json
import os
import sys

# Le coeur du service vit a cote de ce carnet. On le resout explicitement au
# lieu de compter sur le repertoire courant : sous Kernel Gateway, le cwd depend
# de la facon dont le service est lance, et un seed_uri absolu depuis la racine
# du depot laissait le kernel sans module (issue #22439, defaut a). Le lanceur
# run_z3_sudoku_gateway.py, qui connait le dossier du coeur, le publie au
# processus kernel via Z3_SUDOKU_CORE_DIR : c'est le premier candidat lu ici.
# Demarrer kernel_gateway directement (sans le lanceur) depuis un dossier sans
# rapport exige de poser cette variable a la main (voir le README).
CORE_MODULE = "z3_sudoku_core.py"
CHEMIN_DEPUIS_RACINE = os.path.join(
    "MyIA.AI.Notebooks", "SymbolicAI", "SMT", "Z3-API", "gateway"
)


def _chemin_affichable(dossier):
    """Deux derniers segments du dossier : lisible, stable, sans chemin machine.

    La sortie committee ne doit dependre ni de la machine ni du repertoire de
    lancement : on ne garde donc que la fin du chemin.
    """
    morceaux = os.path.abspath(dossier).replace(os.sep, "/").rstrip("/").split("/")
    return "/".join(morceaux[-2:])


def _dossiers_candidats():
    """Dossiers ou le coeur peut se trouver, du plus explicite au plus general."""
    explicite = os.environ.get("Z3_SUDOKU_CORE_DIR")
    if explicite:
        yield explicite
    courant = os.path.abspath(os.getcwd())
    for _ in range(8):
        yield courant
        # Lancement depuis la racine du depot, ou depuis n'importe quel parent :
        # on essaie le chemin du coeur relativement a chaque niveau remonte.
        yield os.path.join(courant, CHEMIN_DEPUIS_RACINE)
        parent = os.path.dirname(courant)
        if parent == courant:
            break
        courant = parent


CORE_EXPLICITE = os.environ.get("Z3_SUDOKU_CORE_DIR")
CORE_DIR = next(
    (d for d in _dossiers_candidats() if os.path.isfile(os.path.join(d, CORE_MODULE))),
    None,
)

# Un echec d'import ne doit pas tuer le kernel : un kernel mort laisse le pool
# vide, et chaque requete pend alors sans reponse (defaut b de #22439). La cause
# est donc conservee ici, et c'est la cellule d'endpoint qui la restitue.
CORE_IMPORT_ERROR = None
if CORE_DIR is None:
    CORE_IMPORT_ERROR = "module %s introuvable ; dossiers essayes : %s" % (
        CORE_MODULE,
        ", ".join(
            dict.fromkeys(_chemin_affichable(d) for d in _dossiers_candidats())
        ),
    )
else:
    sys.path.insert(0, CORE_DIR)
    try:
        from z3_sudoku_core import solve_request
    except Exception as exc:
        # Cause conservee telle quelle, jamais avalee en silence.
        CORE_IMPORT_ERROR = "%s: %s" % (type(exc).__name__, exc)

# REQUEST n'existe que sous Kernel Gateway (mode notebook-http) ; l'objet est
# injecte au moment de chaque requete. Hors gateway : None -> mode local.
REQUEST = globals().get("REQUEST")

# Puzzle facile du carnet phare Z3-02-Sudoku-Python.ipynb (cellule 3).
DEMO_PUZZLE = (
    "530070000"
    "600195000"
    "098000060"
    "800060003"
    "400803001"
    "700020006"
    "060000280"
    "000419005"
    "000080079"
)

print("Mode :", "Kernel Gateway (notebook-http)" if REQUEST is not None else "local (demonstration)")
if CORE_IMPORT_ERROR is None:
    # Un dossier explicite qui ne porte pas le coeur n'est pas une erreur (c'est
    # un candidat, pas une contrainte) -- mais l'operateur qui l'a pose doit
    # savoir qu'il a ete ignore, sinon il croit forcer un coeur precis.
    if (
        CORE_EXPLICITE
        and os.path.abspath(CORE_EXPLICITE) != os.path.abspath(CORE_DIR)
    ):
        print(
            "Attention : Z3_SUDOKU_CORE_DIR ignore (pas de coeur la) ; coeur retenu :",
            _chemin_affichable(CORE_DIR),
        )
    print("Coeur :", _chemin_affichable(CORE_DIR))
else:
    print("Coeur : INDISPONIBLE --", CORE_IMPORT_ERROR)


Mode : local (demonstration)
Coeur : Z3-API/gateway


## Endpoint `POST /sudoku/solve`

La cellule ci-dessous est annotee en premiere ligne par `# POST /sudoku/solve` :
Kernel Gateway l'execute a chaque requete HTTP, avec l'objet `REQUEST` injecte
dans l'espace de noms. Le JSON imprime devient le corps de la reponse.

In [2]:
# POST /sudoku/solve
# Sous Kernel Gateway 3.x, REQUEST est une CHAINE JSON (double json.dumps
# cote serveur au niveau requete) dont la cle "body" contient le corps HTTP
# deja parse en dict ; selon les versions on trouve aussi REQUEST en dict
# direct, ou le corps recu comme chaine JSON brute -- les trois formes sont
# acceptees. Un client qui a enveloppe son payload une fois de trop
# ({"body": "<corps stringify>"}) est deballe aussi : le contrat ne connait
# que la cle "puzzle", la tolerance ne peut pas capturer un vrai puzzle.
if CORE_IMPORT_ERROR is not None:
    # Le service est joignable mais son coeur n'a pas pu etre charge : on rend
    # une erreur structuree et lisible, avec sa cause, au lieu de laisser le
    # client expirer sur un kernel qui ne repondra jamais (defaut b de #22439).
    print(json.dumps({
        "status": "error",
        "error": "kernel core unavailable",
        "detail": CORE_IMPORT_ERROR,
    }))
else:
    requete = None
    if isinstance(REQUEST, str):
        try:
            requete = json.loads(REQUEST)
        except Exception:
            requete = None
    elif isinstance(REQUEST, dict):
        requete = REQUEST
    corps = requete.get("body") if isinstance(requete, dict) else None
    if isinstance(corps, str):
        # Corps recu comme chaine JSON brute (client qui stringify son corps,
        # ou version de gateway qui ne le parse pas). Sans ce decodage, la
        # chaine entiere partait dans solve_request et ressortait en
        # "invalid" propre -- le client croyait son puzzle insoluble.
        try:
            corps = json.loads(corps)
        except Exception:
            corps = None
    if (
        isinstance(corps, dict)
        and "puzzle" not in corps
        and isinstance(corps.get("body"), str)
    ):
        # Enveloppe une fois de trop mesuree sur gateway 3.0.1 (review #22555) :
        # corps = {"body": "{\"puzzle\": ...}"} apres parse de la requete.
        try:
            interne = json.loads(corps["body"])
        except Exception:
            interne = None
        if isinstance(interne, dict):
            corps = interne
    puzzle = corps.get("puzzle") if isinstance(corps, dict) else None
    if REQUEST is None:
        # Execution locale : demonstration sur le puzzle du carnet phare.
        puzzle = DEMO_PUZZLE
    reponse = solve_request(puzzle)
    print(json.dumps(reponse))


{"status": "sat", "solution": "534678912672195348198342567859761423426853791713924856961537284287419635345286179", "time_ms": 75.5}


## Execution locale

Sous Kernel Gateway, cette meme cellule repond aux POST ; en execution locale
elle vient de resoudre le puzzle de demonstration. La variante `unsat` (puzzle
contradictoire) est couverte par les tests unitaires de `test_z3_sudoku_core.py`.